1) Add __pow__ to the Value class so you can compute x ** n. Verify that d/dx(x^3) at x=2 equals 12.0.

In [1]:
class Value:
    def __init__(self, data, _children=(), _op=''):
        self.data = data
        self.grad = 0
        self._backward = lambda: None
        self._prev = set(_children)
        self._op = _op

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other), '+')

        def backward():
            self.grad += out.grad
            other.grad += out.grad

        out._backward = backward
        return out

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other), '*')

        def backward():
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad

        out._backward = backward
        return out

    def __pow__(self, n):
        out = Value(self.data ** n, (self,), f'**{n}')

        def backward():
            self.grad += n * (self.data ** (n - 1)) * out.grad

        out._backward = backward
        return out

    def backward(self):
        topo = []
        visited = set()

        def build(v):
            if v not in visited:
                visited.add(v)
                for child in v._prev:
                    build(child)
                topo.append(v)

        build(self)
        self.grad = 1

        for v in reversed(topo):
            v._backward()


x = Value(2)
y = x ** 3

y.backward()

print(y.data)
print(x.grad)

8
12


2) Add tanh as an activation function. Verify that tanh'(0) = 1 and tanh'(2) = 0.0707 (approx).

In [2]:
import math

class Value:
    def __init__(self, data, _children=(), _op=''):
        self.data = data
        self.grad = 0
        self._backward = lambda: None
        self._prev = set(_children)
        self._op = _op

    def tanh(self):
        t = math.tanh(self.data)
        out = Value(t, (self,), 'tanh')

        def backward():
            # Formula: tanh'(x) = 1 - tanh(x)^2
            self.grad += (1 - t**2) * out.grad

        out._backward = backward
        return out

    def backward(self):
        self.grad = 1
        self._backward()


x = Value(0)
y = x.tanh()
y.backward()
print("tanh'(0) =", x.grad)

x = Value(2)
y = x.tanh()
y.backward()
print("tanh'(2) =", x.grad)

tanh'(0) = 1.0
tanh'(2) = 0.07065082485316443


3) Build a computation graph for a single neuron: y = relu(w1x1 + w2x2 + b). Compute all five gradients and verify against PyTorch.


In [3]:
import torch

class Value:
    def __init__(self, data, _children=(), _op=''):
        self.data = data
        self.grad = 0
        self._backward = lambda: None
        self._prev = set(_children)
        self._op = _op

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other), '+')

        def backward():
            self.grad += out.grad
            other.grad += out.grad

        out._backward = backward
        return out

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other), '*')

        def backward():
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad

        out._backward = backward
        return out

    def relu(self):
        out = Value(max(0, self.data), (self,), 'ReLU')

        def backward():
            self.grad += (1 if self.data > 0 else 0) * out.grad

        out._backward = backward
        return out

    def backward(self):
        topo = []
        visited = set()

        def build(v):
            if v not in visited:
                visited.add(v)
                for child in v._prev:
                    build(child)
                topo.append(v)

        build(self)
        self.grad = 1

        for v in reversed(topo):
            v._backward()


x1 = Value(2)
x2 = Value(3)
w1 = Value(4)
w2 = Value(5)
b = Value(1)

z = w1 * x1 + w2 * x2 + b
y = z.relu()

y.backward()

print("Our gradients:")
print("dw1 =", w1.grad)
print("dw2 =", w2.grad)
print("dx1 =", x1.grad)
print("dx2 =", x2.grad)
print("db  =", b.grad)


x1_t = torch.tensor(2.0, requires_grad=True)
x2_t = torch.tensor(3.0, requires_grad=True)
w1_t = torch.tensor(4.0, requires_grad=True)
w2_t = torch.tensor(5.0, requires_grad=True)
b_t = torch.tensor(1.0, requires_grad=True)

z_t = w1_t * x1_t + w2_t * x2_t + b_t
y_t = torch.relu(z_t)

y_t.backward()

print("\nPyTorch gradients:")
print("dw1 =", w1_t.grad.item())
print("dw2 =", w2_t.grad.item())
print("dx1 =", x1_t.grad.item())
print("dx2 =", x2_t.grad.item())
print("db  =", b_t.grad.item())

ModuleNotFoundError: No module named 'torch'